# Lab · Huấn luyện hồi quy tuyến tính bằng gradient descent · LỜI GIẢI

> Chỉ mở file này khi bạn đã thử bài tập, hoặc đã dùng hết cả 3 gợi ý.

Bài tập cho [`lesson.md`](../lesson.md) (Module 3, Bài 2). Theo lab "Linear regression 1D: Training Two Parameter" của khóa: tạo dữ liệu có nhiễu, huấn luyện `w` và `b`, thử các learning rate khác nhau.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch", torch.__version__)

## Bài 1. Tạo dữ liệu có nhiễu

Đặt `torch.manual_seed(0)`. Rồi:
- `X = torch.arange(-3, 3, 0.1).view(-1, 1)`: các giá trị từ −3 đến 3, cột;
- `f = 1 * X - 1`: đường thẳng thật (độ dốc 1, bias −1);
- `Y = f + 0.1 * torch.randn(X.size())`: thêm nhiễu Gauss.

Vẽ `Y` (dấu `x` đỏ) và `f` (đường) lên cùng một đồ thị.

In [ ]:
torch.manual_seed(0)
X = torch.arange(-3, 3, 0.1).view(-1, 1)
f = 1 * X - 1
Y = f + 0.1 * torch.randn(X.size())
plt.plot(X.numpy(), Y.numpy(), "rx", label="y (có nhiễu)")
plt.plot(X.numpy(), f.numpy(), label="f (đường thật)")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

In [ ]:
assert tuple(X.shape) == (60, 1) and tuple(Y.shape) == (60, 1)
assert (Y - f).abs().max().item() < 0.5, "nhiễu phải nhỏ (độ lệch chuẩn 0.1)"
assert (Y - f).abs().max().item() > 0, "phải có nhiễu"
print("✅ Bài 1 đúng")

## Bài 2. Mô hình và hàm cost

Đặt tham số khởi đầu `w = -15.0`, `b = -10.0` (có theo dõi gradient).
Viết `forward(x)` trả về `w * x + b` và `criterion(yhat, y)` trả về MSE (`torch.mean((yhat - y) ** 2)`).
Lưu cost ban đầu `loss0 = criterion(forward(X), Y)`.

In [ ]:
w = torch.tensor(-15.0, requires_grad=True)
b = torch.tensor(-10.0, requires_grad=True)

def forward(x):
    return w * x + b

def criterion(yhat, y):
    return torch.mean((yhat - y) ** 2)

loss0 = criterion(forward(X), Y)
print(loss0)

In [ ]:
assert w.requires_grad and b.requires_grad
assert abs(loss0.item() - 835.7) < 1.0, "cost ban đầu khoảng 835.7"
print("✅ Bài 2 đúng: tham số khởi đầu còn rất xa tham số thật")

## Bài 3. Gradient: để autograd tính và kiểm tra bằng công thức

- Tính cost, gọi `backward()`, rồi lưu `gw = w.grad.item()` và `gb = b.grad.item()`.
- `gw_manual`, `gb_manual`: gradient tính bằng **công thức tay** của MSE:
  `∂/∂w = 2 · mean((ŷ − y) · x)`, `∂/∂b = 2 · mean(ŷ − y)`.
  (Dùng `torch.no_grad()` để không ghi vào đồ thị.)
- Hai cách phải cho cùng kết quả. Sau đó **đặt gradient về 0**.

In [ ]:
loss = criterion(forward(X), Y)
loss.backward()
gw = w.grad.item()
gb = b.grad.item()

with torch.no_grad():
    yhat = forward(X)
    gw_manual = (2 * ((yhat - Y) * X).mean()).item()
    gb_manual = (2 * (yhat - Y).mean()).item()

w.grad.zero_()
b.grad.zero_()
print(gw, gw_manual, gb, gb_manual)

In [ ]:
assert abs(gw - gw_manual) < 1e-3 and abs(gb - gb_manual) < 1e-3, "autograd và công thức tay phải khớp"
assert abs(gw + 95.19) < 0.1 and abs(gb + 16.41) < 0.1
assert w.grad.item() == 0 and b.grad.item() == 0, "gradient phải được đặt về 0"
print("✅ Bài 3 đúng: gradient âm → tăng w và b sẽ làm cost giảm")

## Bài 4. Một bước cập nhật bằng tay

Với `lr = 0.1` và gradient `gw`, `gb` ở bài 3, tính tham số sau **một** bước:
`w_new = w − lr · gw`, `b_new = b − lr · gb` (dưới dạng số Python, **chưa** đụng vào tensor `w`, `b`).
Rồi kiểm tra cost sau khi đổi sang tham số mới có **giảm** không (`improved`).

In [ ]:
lr = 0.1
w_new = w.item() - lr * gw
b_new = b.item() - lr * gb
new_cost = torch.mean((w_new * X + b_new - Y) ** 2)
improved = new_cost.item() < loss0.item()
print(w_new, b_new, improved)

In [ ]:
assert abs(w_new - (-15 + 9.519)) < 0.02 and abs(b_new - (-10 + 1.641)) < 0.02
assert improved is True
print("✅ Bài 4 đúng")

## Bài 5. Viết vòng lặp huấn luyện

Viết hàm `train(lr, iterations)`:
- tạo tham số riêng `w = -15.0`, `b = -10.0` (có gradient) bên trong hàm;
- lặp `iterations` lần theo **năm bước**: dự đoán, tính loss (và lưu `loss.item()` vào list `LOSS`), `backward()`, cập nhật dưới `torch.no_grad()`, đặt gradient về 0;
- trả về `(w.item(), b.item(), LOSS)`.

Chạy `train(0.1, 15)` và lưu `w1, b1, LOSS1`. **Đoán trước** `w` sau 15 vòng gần bao nhiêu (tham số thật là 1 và −1).

In [ ]:
def train(lr, iterations):
    w = torch.tensor(-15.0, requires_grad=True)
    b = torch.tensor(-10.0, requires_grad=True)
    LOSS = []
    for epoch in range(iterations):
        loss = criterion(w * X + b, Y)
        LOSS.append(loss.item())
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    return w.item(), b.item(), LOSS

w1, b1, LOSS1 = train(0.1, 15)
print(w1, b1, LOSS1[:3])

In [ ]:
assert len(LOSS1) == 15
assert all(LOSS1[i] >= LOSS1[i + 1] for i in range(14)), "loss phải giảm dần"
assert LOSS1[0] > 800 and LOSS1[-1] < 0.5
assert abs(w1 - 1.0) < 0.1, "w phải gần độ dốc thật = 1"
print("✅ Bài 5 đúng: bạn vừa huấn luyện xong mô hình đầu tiên")

## Bài 6. Luyện tập: learning rate 0.2

Huấn luyện lại với `lr = 0.2`, cùng 15 vòng. Lưu `w2, b2, LOSS2`.
Rồi vẽ `LOSS1` và `LOSS2` trên cùng một đồ thị để so sánh. Learning rate nào hội tụ nhanh hơn?

In [ ]:
w2, b2, LOSS2 = train(0.2, 15)
plt.plot(LOSS1, label="lr = 0.1")
plt.plot(LOSS2, label="lr = 0.2")
plt.xlabel("epoch"); plt.ylabel("cost"); plt.legend(); plt.show()
print(w2, b2, LOSS2[-1])

In [ ]:
assert LOSS2[-1] < LOSS1[-1], "lr = 0.2 hội tụ nhanh hơn lr = 0.1 trong 15 vòng"
assert abs(w2 - 1.0) < 0.05 and abs(b2 + 1.0) < 0.05
print("✅ Bài 6 đúng")

## Bài 7. Learning rate quá lớn

Huấn luyện với `lr = 0.4` trong 15 vòng. Lưu `LOSS3`.
**Đoán trước** điều gì sẽ xảy ra. Vẽ đường loss (dùng trục log nếu cần: `plt.yscale("log")`). Lưu `diverged = LOSS3[-1] > LOSS3[0]`.

In [ ]:
_, _, LOSS3 = train(0.4, 15)
diverged = LOSS3[-1] > LOSS3[0]
plt.plot(LOSS3); plt.yscale("log")
plt.xlabel("epoch"); plt.ylabel("cost (log)"); plt.show()
print(diverged, LOSS3[-1])

In [ ]:
assert diverged is True, "lr quá lớn làm cost tăng thay vì giảm"
print("✅ Bài 7 đúng: learning rate phải vừa phải")

## Bài 8. Vẽ đường thẳng đã học

Vẽ dữ liệu `Y` (dấu `x` đỏ), đường thật `f`, và đường thẳng mô hình học được với `lr = 0.2` (dùng `w2`, `b2`) lên cùng một đồ thị.
Lưu `slope_err = abs(w2 - 1.0)`.

In [ ]:
slope_err = abs(w2 - 1.0)
plt.plot(X.numpy(), Y.numpy(), "rx", label="y")
plt.plot(X.numpy(), f.numpy(), label="f (thật)")
plt.plot(X.numpy(), (w2 * X + b2).numpy(), "--", label="mô hình (lr = 0.2)")
plt.legend(); plt.show()
print(slope_err)

In [ ]:
assert slope_err < 0.05
print("✅ Bài 8 đúng")

## Bài 9. Thử thách: dừng sớm theo loss

Viết `train_until(lr, tol, max_epochs)`: huấn luyện như trước nhưng **dừng** khi mức giảm loss giữa hai vòng liên tiếp nhỏ hơn `tol`, hoặc khi đạt `max_epochs`.
Trả về `(số epoch đã chạy, loss cuối)`. Gọi với `lr=0.2, tol=1e-4, max_epochs=1000`, lưu `n_epochs, final_loss`.

In [ ]:
def train_until(lr, tol, max_epochs):
    w = torch.tensor(-15.0, requires_grad=True)
    b = torch.tensor(-10.0, requires_grad=True)
    prev = None
    for epoch in range(1, max_epochs + 1):
        loss = criterion(w * X + b, Y)
        cur = loss.item()
        if prev is not None and prev - cur < tol:
            return epoch, cur
        prev = cur
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    return max_epochs, cur

n_epochs, final_loss = train_until(0.2, 1e-4, 1000)
print(n_epochs, final_loss)

In [ ]:
assert 5 < n_epochs < 1000, "phải dừng sớm, không chạy hết 1000 vòng"
assert final_loss < 0.02
print("✅ Bài 9 đúng: dừng khi loss không còn giảm đáng kể")

## Hết